# SOXX 2019 旧表单原件来源

按 `plan.json` 冻结两份正文原件，先取得精确字节、响应头并完成 NAS 远端哈希与限定恢复；随后才审查 SOXX 投资明细身份。

In [1]:
import json,hashlib,subprocess,time,tarfile,tempfile,shlex
from pathlib import Path
from datetime import datetime,timezone
from src.data.nas_market_data import _settings,nas_api_call
root=Path.cwd();card=root/'research/stage1_soxx_2019_transition_bodies_20261023';plan=json.loads((card/'plan.json').read_text())
def sha(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()
assert plan['frozen'] and not (card/'attempt_started.json').exists()
assert sha(root/plan['parent_index_identity'])==plan['parent_index_identity_sha256']
raw=root/'data/snapshots/stage1_soxx_2019_transition_bodies_20261023';raw.mkdir(parents=True,exist_ok=True);assert not any(raw.iterdir())
state={'status':'started','body_GET_attempts':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'attempt_started.json').write_text(json.dumps(state,indent=2)+'\n')
receipts=[];error=None
for i,spec in enumerate(plan['exact_bodies']):
 state['body_GET_attempts']+=1;(card/'attempt_started.json').write_text(json.dumps(state,indent=2)+'\n')
 body=raw/(spec['accession']+'.html');headers=raw/(spec['accession']+'.headers');started=datetime.now(timezone.utc).isoformat()
 cmd=['curl','--http1.1','--noproxy','*','--connect-timeout',str(plan['request']['connect_timeout_seconds']),'--max-time',str(plan['request']['total_timeout_seconds_each']),'--max-filesize',str(plan['request']['max_response_bytes_each']),'--fail-with-body','-sS','-L','-D',str(headers),'-o',str(body),'-A',plan['request']['user_agent'],'-w','%{http_code} %{url_effective}',spec['url']]
 try:
  run=subprocess.run(cmd,capture_output=True,text=True,timeout=210);fields=run.stdout.rsplit(' ',1);status=int(fields[0]) if len(fields)==2 and fields[0].isdigit() else None
  receipt={'accession':spec['accession'],'report_period':spec['report_period'],'url':spec['url'],'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'curl_exit_code':run.returncode,'http_status':status,'final_url':fields[-1] if fields else None,'stderr':run.stderr[:300]}
 except Exception as exc:receipt={'accession':spec['accession'],'report_period':spec['report_period'],'url':spec['url'],'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'transport_exception':type(exc).__name__+':'+str(exc)[:180]}
 receipt.update(raw_sha256=sha(body) if body.exists() else None,raw_bytes=body.stat().st_size if body.exists() else 0,headers_sha256=sha(headers) if headers.exists() else None,headers_bytes=headers.stat().st_size if headers.exists() else 0)
 receipts.append(receipt)
 if receipt.get('curl_exit_code')!=0 or receipt.get('http_status')!=200 or not body.exists() or body.stat().st_size>plan['request']['max_response_bytes_each']:error='transport_or_size_failure';break
 if i+1<len(plan['exact_bodies']):time.sleep(plan['request']['minimum_seconds_between_requests'])
(card/'capture_receipts.json').write_text(json.dumps(receipts,indent=2)+'\n')
manifest={'plan_sha256':sha(card/'plan.json'),'files':[{'name':p.name,'sha256':sha(p),'bytes':p.stat().st_size} for p in sorted(raw.iterdir()) if p.is_file()],'capture_receipts_sha256':sha(card/'capture_receipts.json')};(card/'source_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
settings=_settings(root)
with tempfile.TemporaryDirectory(prefix='soxx-2019-body-') as temp:
 archive=Path(temp)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as out:
  for x in manifest['files']:out.add(raw/x['name'],arcname='raw/'+x['name'])
  out.add(card/'source_manifest.json',arcname='source_manifest.json');out.add(card/'capture_receipts.json',arcname='capture_receipts.json')
 digest=sha(archive);evidence=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact']
 assert evidence['sha256']==digest and evidence['size_bytes']==archive.stat().st_size
 remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 with tempfile.TemporaryDirectory(prefix='soxx-2019-body-restore-') as rd:
  restored=Path(rd)/'restore.tar.gz'
  with restored.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,timeout=150)
  assert run.returncode==0 and sha(restored)==digest
  with tarfile.open(restored,'r:gz') as source:
   for x in manifest['files']:
    member=source.extractfile('raw/'+x['name']);assert member and hashlib.sha256(member.read()).hexdigest()==x['sha256']
result={'status':'two_exact_2019_old_form_bodies_archived_identity_pending' if not error and len(receipts)==2 else 'partial_body_transport_needs_repair','error':error,'body_GET_attempts':state['body_GET_attempts'],'HTTP200_bodies':sum(x.get('http_status')==200 and x.get('curl_exit_code')==0 for x in receipts),'nas_archive_sha256':digest,'nas_archive_bytes':evidence['size_bytes'],'nas_remote_and_scoped_restore_verified':True,'market_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'transport_result.json').write_text(json.dumps(result,indent=2)+'\n')
print(result)


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'status': 'two_exact_2019_old_form_bodies_archived_identity_pending', 'error': None, 'body_GET_attempts': 2, 'HTTP200_bodies': 2, 'nas_archive_sha256': 'bf6f45614df9703c9c586133bf53be61c8008d4496ea09693671057cec9c43c9', 'nas_archive_bytes': 4180843, 'nas_remote_and_scoped_restore_verified': True, 'market_outcome_reads': 0, 'model_fits': 0, 'native_accounts': 0}


## 中文结论

两份索引指定的原始 SEC 正文均 HTTP 200，原字节及响应头已 NAS 远端哈希和限定恢复。零网络原文扫描在 2018-12-31 和 2019-03-31 各找到一处 SOXX 自身的日期、Schedule of Investments、Security/Shares/Value/Common Stocks 连续表头。仅证明原件身份；尚未提取或闭合普通股数值，0 收益读取、0 拟合、0 账户。下一步在本原件上冻结数值解析。